In [1]:
import os
import sys
# current_dir = os.path.dirname(os.path.abspath(__file__))
# parent_dir = os.path.dirname(current_dir)
sys.path.append("../")
# import numpy as np
# from statistics import mode
# import matplotlib.pyplot as plt
# from datetime import datetime, timedelta
import requests
import baostock as bs
import akshare as ak
import pandas as pd
import yfinance as yf
import time
import datetime
from Config.Config import ETF_DATA_PATH
from Utils.main import PrintUtils, FileUtils, JsonUtils, TextUtils
THREAD_SAFE_PRINT = PrintUtils.THREAD_SAFE_PRINT
Check_Folder = FileUtils.Check_Folder
Check_File = FileUtils.Check_File
Dict_to_JsonFile = JsonUtils.Dict_to_JsonFile
JsonFile_to_Dict = JsonUtils.JsonFile_to_Dict
Format_Num = TextUtils.Format_Num

# Get Benchmark Index

- HS 300: 000300.SH
- CS 500: 000905.SH
- CS 1000: 000852.SH

In [5]:
# Define Indices (Code format for Baostock usually includes 'sh.' or 'sz.')
BENCHMARK_INDICES = {
    'HS300': 'sh.000300',
    'CSI500': 'sh.000905', 
    'CSI1000': 'sh.000852'
}

def Download_Benchmark_Index_Data(etf_folder_path, benchmark_code_dict, benchmark_path_mapping_path, Log_File_Path=""):
    """
    - Batch fetch Benchmark Index data (HS300, CSI500, etc.) using Baostock.
    - Matches the logic of Download_SW_Index_Data: checks existing files, updates mapping JSON, logs status.
    - benchmark_code_dict is like: {'HS300': 'sh.000300', 'CSI500': 'sh.000905'}
    """

    # 1. Check existing files (File format: "BM-index_code-start_date-end_date.csv")
    exist_code_list = []
    
    # Ensure folder exists
    if not os.path.exists(etf_folder_path):
        os.makedirs(etf_folder_path)

    for filename in os.listdir(etf_folder_path):
        if filename.startswith("Benchmark") and filename.endswith(".csv"):
            # Filename structure assumption: BM-sh.000300-20000101-20231231.csv
            try:
                parts = filename.split("-")
                if len(parts) >= 2:
                    existing_code = parts[1]
                    exist_code_list.append(existing_code)
            except IndexError:
                continue
                
    if exist_code_list: 
        THREAD_SAFE_PRINT("Download Benchmark", f"Existing index data found for: {exist_code_list}", Log_File_Path)

    # 2. Login to Baostock
    lg = bs.login()
    if lg.error_code != '0':
        THREAD_SAFE_PRINT("Download Benchmark", f"❌ Login failed: {lg.error_msg}", Log_File_Path)
        return

    # Ensure mapping file exists
    Check_File(benchmark_path_mapping_path)
    benchmark_path_mapping = JsonFile_to_Dict(benchmark_path_mapping_path)

    # 3. Iterate and Fetch
    for index_name, index_code in benchmark_code_dict.items():
        try:
            if index_code not in exist_code_list:
                THREAD_SAFE_PRINT("Download Benchmark", f"Starting to fetch {index_name} ({index_code})...", Log_File_Path)
                
                # Query Baostock
                # frequency: d=daily
                # adjustflag: 3=no adjust (standard for indices)
                rs = bs.query_history_k_data_plus(
                    code=index_code,
                    fields="date,code,open,high,low,close,preclose,volume,amount,pctChg",
                    start_date='1990-01-01', # Far enough back to catch all history
                    end_date=datetime.datetime.now().strftime('%Y-%m-%d'),
                    frequency="d",
                    adjustflag="3"
                )

                if rs.error_code == '0':
                    data_list = []
                    while rs.next():
                        data_list.append(rs.get_row_data())
                    
                    if not data_list:
                        THREAD_SAFE_PRINT("Download Benchmark", f"⚠️ No data returned for {index_name}", Log_File_Path)
                        continue

                    # Create DataFrame
                    df = pd.DataFrame(data_list, columns=rs.fields)
                    
                    # 4. Data Cleaning (Standardizing to match SW Index format)
                    df['date'] = pd.to_datetime(df['date'])
                    
                    # Convert numeric columns
                    numeric_cols = ['open', 'high', 'low', 'close', 'preclose', 'volume', 'amount', 'pctChg']
                    for col in numeric_cols:
                        df[col] = pd.to_numeric(df[col], errors='coerce')

                    # Rename columns to match your other function's output standard
                    # 'amount' in Baostock is usually 'turnover' in other APIs
                    df = df.rename(columns={'code': 'index_code', 'amount': 'turnover'})
                    
                    df = df.set_index('date')
                    df = df.sort_index()

                    start_date = df.index.min().strftime("%Y%m%d")
                    end_date = df.index.max().strftime("%Y%m%d")

                    # 5. Save to CSV and Update Mapping
                    # Using "BM" prefix to distinguish from "SW" files
                    csv_filename = f"Benchmark-{index_code}-{start_date}-{end_date}.csv"
                    
                    # Update Mapping Dict
                    benchmark_path_mapping[index_name] = csv_filename
                    Dict_to_JsonFile(benchmark_path_mapping, benchmark_path_mapping_path)
                    # Save File
                    df.to_csv(os.path.join(etf_folder_path, csv_filename))
                    
                    THREAD_SAFE_PRINT("Download Benchmark", f"✅ Successfully fetched: {index_name} ({start_date} to {end_date})", Log_File_Path)
                    THREAD_SAFE_PRINT("Download Benchmark", f"Saved to: {csv_filename}", Log_File_Path)
                    
                else:
                    THREAD_SAFE_PRINT("Download Benchmark", f"❌ Baostock Error for {index_name}: {rs.error_msg}", Log_File_Path)
        except Exception as e:
            THREAD_SAFE_PRINT("Download Benchmark", f"❌ Failed to fetch {index_name}: {e}", Log_File_Path)
    # 6. Logout
    bs.logout()
    THREAD_SAFE_PRINT("Download Benchmark", "Process completed. Logged out.", Log_File_Path)

# Define Paths
# ETF_DATA_PATH should be defined in your config
BENCHMARK_PATH_MAPPING_PATH = ETF_DATA_PATH + "BENCHMARK-PATH-MAPPING.json"

Download_Benchmark_Index_Data(etf_folder_path=ETF_DATA_PATH, benchmark_code_dict=BENCHMARK_INDICES, benchmark_path_mapping_path=BENCHMARK_PATH_MAPPING_PATH)

login success!
[26/1/31 20:35:9][Download Benchmark] Starting to fetch HS300 (sh.000300)...
[26/1/31 20:35:15][Download Benchmark] ✅ Successfully fetched: HS300 (20050104 to 20260130)
[26/1/31 20:35:15][Download Benchmark] Saved to: Benchmark-sh.000300-20050104-20260130.csv
[26/1/31 20:35:15][Download Benchmark] Starting to fetch CSI500 (sh.000905)...
[26/1/31 20:35:21][Download Benchmark] ✅ Successfully fetched: CSI500 (20050104 to 20260130)
[26/1/31 20:35:21][Download Benchmark] Saved to: Benchmark-sh.000905-20050104-20260130.csv
[26/1/31 20:35:21][Download Benchmark] Starting to fetch CSI1000 (sh.000852)...
[26/1/31 20:35:28][Download Benchmark] ✅ Successfully fetched: CSI1000 (20050104 to 20260130)
[26/1/31 20:35:28][Download Benchmark] Saved to: Benchmark-sh.000852-20050104-20260130.csv
logout success!
[26/1/31 20:35:28][Download Benchmark] Process completed. Logged out.


# Get Industry Index

In [2]:
# Obtain sw first industry list
# sw_index_df = ak.sw_index_first_info()
# print(sw_index_df.head())
# sw_index_df

In [3]:
# Obtain sw second industry list
# sw_index_second_df = ak.sw_index_second_info()
# sw_index_second_df

In [4]:
# Obtain sw thrid industry list
# sw_index_thrid_df = ak.sw_index_third_info()
# sw_index_thrid_df

In [5]:
def Build_SW_Sector_Mapping(sw_index_first_df, sw_index_second_df, sw_index_thrid_df):
    """
    - `sw_index_first_df`, `sw_index_second_df` and `sw_index_thrid_df` are from akshare
    - The output is like {"industry": {"level": ..., "index_code": ...}, ...}
    """
    SW_SECTOR_MAPPING = {}
    
    # ---------------------------------------------------------
    # 1. Process Level 1 (First Class)
    # ---------------------------------------------------------
    # Key format: "L1"
    for _, row in sw_index_first_df.iterrows():
        name = row['行业名称']
        code = row['行业代码']
        
        SW_SECTOR_MAPPING[name] = {
            "level": 1, 
            "index_code": code.split(".")[0]
        }

    # ---------------------------------------------------------
    # 2. Process Level 2 (Second Class)
    # ---------------------------------------------------------
    # We need a helper dict to look up the Grandparent (L1) when we get to Level 3.
    # Map: Level 2 Name -> Level 1 Name
    l2_to_l1_name_map = {}

    for _, row in sw_index_second_df.iterrows():
        l2_name = row['行业名称']
        l1_name = row['上级行业']  # This is the L1 Name
        code = row['行业代码']
        
        # 1. Construct Key: "L1-L2"
        key = f"{l1_name}-{l2_name}"
        
        # 2. Add to result
        SW_SECTOR_MAPPING[key] = {
            "level": 2, 
            "index_code": code.split(".")[0]
        }
        
        # 3. Store the relationship for the next step
        l2_to_l1_name_map[l2_name] = l1_name

    # ---------------------------------------------------------
    # 3. Process Level 3 (Third Class)
    # ---------------------------------------------------------
    for _, row in sw_index_thrid_df.iterrows():
        l3_name = row['行业名称']
        l2_name = row['上级行业'] # This is the L2 Name
        code = row['行业代码']
        
        # Retrieve the Grandparent (L1) name using the helper dict
        l1_name = l2_to_l1_name_map.get(l2_name)
        
        if l1_name:
            # Construct Key: "L1-L2-L3"
            key = f"{l1_name}-{l2_name}-{l3_name}"
            
            SW_SECTOR_MAPPING[key] = {
                "level": 3, 
                "index_code": code.split(".")[0]
            }
        else:
            print(f"Warning: Could not find Level 1 parent for Level 2 sector '{l2_name}'")

    return SW_SECTOR_MAPPING

# SW_SECTOR_MAPPING = Build_SW_Sector_Mapping(sw_index_df, sw_index_second_df, sw_index_thrid_df)
# SW_SECTOR_MAPPING

In [6]:
# Dict_to_JsonFile(SW_SECTOR_MAPPING, ETF_DATA_PATH + "SW-SECTOR-CODE-MAPPING.json")
SW_SECTOR_MAPPING = JsonFile_to_Dict(ETF_DATA_PATH + "SW-SECTOR-CODE-MAPPING.json")
SW_SECTOR_INDEX = {k: v['index_code'] for k, v in SW_SECTOR_MAPPING.items()}
SW_SECTOR_INDEX

{'农林牧渔': '801010',
 '基础化工': '801030',
 '钢铁': '801040',
 '有色金属': '801050',
 '电子': '801080',
 '汽车': '801880',
 '家用电器': '801110',
 '食品饮料': '801120',
 '纺织服饰': '801130',
 '轻工制造': '801140',
 '医药生物': '801150',
 '公用事业': '801160',
 '交通运输': '801170',
 '房地产': '801180',
 '商贸零售': '801200',
 '社会服务': '801210',
 '银行': '801780',
 '非银金融': '801790',
 '综合': '801230',
 '建筑材料': '801710',
 '建筑装饰': '801720',
 '电力设备': '801730',
 '机械设备': '801890',
 '国防军工': '801740',
 '计算机': '801750',
 '传媒': '801760',
 '通信': '801770',
 '煤炭': '801950',
 '石油石化': '801960',
 '环保': '801970',
 '美容护理': '801980',
 '农林牧渔-种植业': '801016',
 '农林牧渔-渔业': '801015',
 '农林牧渔-饲料': '801014',
 '农林牧渔-农产品加工': '801012',
 '农林牧渔-养殖业': '801017',
 '农林牧渔-动物保健Ⅱ': '801018',
 '基础化工-化学原料': '801033',
 '基础化工-化学制品': '801034',
 '基础化工-化学纤维': '801032',
 '基础化工-塑料': '801036',
 '基础化工-橡胶': '801037',
 '基础化工-农化制品': '801038',
 '基础化工-非金属材料Ⅱ': '801039',
 '钢铁-冶钢原料': '801043',
 '钢铁-普钢': '801044',
 '钢铁-特钢Ⅱ': '801045',
 '有色金属-金属新材料': '801051',
 '有色金属-工业金属': '801055',
 '有色金属-贵金属': 

In [7]:
# ak.index_hist_sw(symbol="801780", period="day")

In [8]:
# Obtain sw index offcially

# # 1. The Target URL (constructed from Host + Path)
# url = "https://www.swsresearch.com/institute-sw/api/index_publish/trend/"

# # 2. Query Parameters (from the URL string)
# params = {
#     "swindexcode": "801180",
#     "period": "DAY"
# }

# # 3. Headers (Mapped from your raw request)
# # Note: I've included the custom headers like 'clientType' and 'token' 
# # as servers often reject requests without them.
# headers = {
#     "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/139.0.0.0 Safari/537.36 Edg/139.0.0.0",
#     "Accept": "*/*",
#     "Accept-Language": "en,en-US;q=0.9,zh-CN;q=0.8,zh;q=0.7,ru;q=0.6",
#     "Referer": "https://www.swsresearch.com/",  # Added commonly required header
#     "clientType": "4",
#     "X-Access-Token": "null", # Sent as a string based on your raw log
#     "token": "",              # Appears empty in your raw log
#     "sec-ch-ua": '"Not;A=Brand";v="99", "Microsoft Edge";v="139", "Chromium";v="139"',
#     "sec-ch-ua-mobile": "?0",
#     "sec-ch-ua-platform": '"Windows"'
# }

# # 4. Cookies
# cookies = {
#     "i18next": "zh-CN"
# }

# try:
#     # Make the GET request
#     response = requests.get(
#         url, 
#         params=params, 
#         headers=headers, 
#         cookies=cookies,
#     )

#     # Check if the request was successful
#     response.raise_for_status()

#     # Print the JSON result
#     data = response.json()
#     print("Request Successful!")
#     print(data)

# except requests.exceptions.RequestException as e:
#     print(f"An error occurred: {e}")

In [9]:
# SECTOR_MAPPING = {
    # '食品饮料': {'index_code': '801120', 'etf_code': '515170'}, # 食品饮料ETF
    # '医药生物': {'index_code': '801150', 'etf_code': '512010'}, # 医药ETF
    # '电子':     {'index_code': '801080', 'etf_code': '512480'}, # 半导体ETF (近似)
    # '计算机':   {'index_code': '801750', 'etf_code': '512760'}, # 芯片ETF/计算机ETF
    # '国防军工': {'index_code': '801740', 'etf_code': '512660'}, # 军工ETF
    # '银行':     {'index_code': '801192', 'etf_code': '512800'}, # 银行ETF
    # '非银金融': {'index_code': '801193', 'etf_code': '512070'}, # 券商ETF
    # '房地产':   {'index_code': '801180', 'etf_code': '512200'}, # 地产ETF
    # '有色金属': {'index_code': '801050', 'etf_code': '512400'}, # 有色60ETF
    # '家用电器': {'index_code': '801110', 'etf_code': '159996'}, # 家电ETF
    # '汽车':     {'index_code': '801880', 'etf_code': '516110'}, # 汽车ETF
    # '电力设备': {'index_code': '801730', 'etf_code': '159915'}, # 创业板(新能源主要载体) 或 光伏ETF
    # ... 
# }

"""
Dictionary structure description:

Key: Industry name (industry field from your advertising data)
Value: {
    'level': Industry level (1=primary, 2=secondary),
    'index_code': Shenwan index code (used for historical data backtesting with akshare),
    'etf_code': Leading ETF code (used for actual trading),
    'etf_name': ETF name (for manual verification)
}
Note: For industries without suitable ETFs, 'etf_code' is set to None.
"""

# SW_SECTOR_MAPPING = {
#     # ==============================
#     # Cyclical Upstream (Resources/Materials)
#     # ==============================
#     '煤炭': {
#         'level': 1, 'index_code': '801950', 
#         'etf_code': '515220', 'etf_name': '煤炭ETF'
#     },
#     '石油石化': {
#         'level': 1, 'index_code': '801960', 
#         'etf_code': '561360', 'etf_name': '石化ETF'
#     },
#     '有色金属': {
#         'level': 1, 'index_code': '801050', 
#         'etf_code': '512400', 'etf_name': '有色60ETF'
#     },
#     '钢铁': {
#         'level': 1, 'index_code': '801040', 
#         'etf_code': '515210', 'etf_name': '钢铁ETF'
#     },
#     '基础化工': {
#         'level': 1, 'index_code': '801030', 
#         'etf_code': '516020', 'etf_name': '化工ETF'
#     },
#     '建筑材料': {
#         'level': 1, 'index_code': '801710', 
#         'etf_code': '159745', 'etf_name': '建材ETF'
#     },

#     # ==============================
#     # Advanced Manufacturing (Hard Tech/High-end Manufacturing)
#     # ==============================
#     '电力设备': { # Includes photovoltaics, wind power, batteries; core of new energy
#         'level': 1, 'index_code': '801730', 
#         'etf_code': '159915', 'etf_name': '创业板50(新能源替代)' # or 516160 新能源ETF
#     },
#     '国防军工': {
#         'level': 1, 'index_code': '801740', 
#         'etf_code': '512660', 'etf_name': '军工ETF'
#     },
#     '机械设备': {
#         'level': 1, 'index_code': '801890', 
#         'etf_code': '516960', 'etf_name': '机械ETF' # Many robotics concepts are here
#     },
#     '汽车': {
#         'level': 1, 'index_code': '801880', 
#         'etf_code': '516110', 'etf_name': '汽车ETF' # or 159806 新能车ETF
#     },

#     # ==============================
#     # Technology TMT
#     # ==============================
#     '电子': {
#         'level': 1, 'index_code': '801080', 
#         'etf_code': '515260', 'etf_name': '电子ETF'
#     },
#     '计算机': {
#         'level': 1, 'index_code': '801750', 
#         'etf_code': '512760', 'etf_name': '芯片ETF' # Computers are often highly correlated with semiconductors
#     },
#     '通信': {
#         'level': 1, 'index_code': '801770', 
#         'etf_code': '515050', 'etf_name': '5G通信ETF'
#     },
#     '传媒': {
#         'level': 1, 'index_code': '801760', 
#         'etf_code': '512980', 'etf_name': '传媒ETF' # Includes gaming, AI applications
#     },

#     # ==============================
#     # Major Consumption
#     # ==============================
#     '食品饮料': {
#         'level': 1, 'index_code': '801120', 
#         'etf_code': '515170', 'etf_name': '食品饮料ETF'
#     },
#     '医药生物': {
#         'level': 1, 'index_code': '801150', 
#         'etf_code': '512010', 'etf_name': '医药ETF'
#     },
#     '家用电器': {
#         'level': 1, 'index_code': '801110', 
#         'etf_code': '159996', 'etf_name': '家电ETF'
#     },
#     '农林牧渔': {
#         'level': 1, 'index_code': '801010', 
#         'etf_code': '159825', 'etf_name': '农业ETF'
#     },
#     '美容护理': {
#         'level': 1, 'index_code': '801980', 
#         'etf_code': None, 'etf_name': None # No pure ETF yet; often replaced with medical aesthetics
#     },
#     '纺织服饰': {
#         'level': 1, 'index_code': '801130', 
#         'etf_code': None, 'etf_name': None # Poor liquidity, recommend avoiding
#     },
#     '轻工制造': {
#         'level': 1, 'index_code': '801140', 
#         'etf_code': None, 'etf_name': None
#     },
#     '商贸零售': {
#         'level': 1, 'index_code': '801200', 
#         'etf_code': None, 'etf_name': None # Lacks representative ETF
#     },
#     '社会服务': { # 旅游、酒店、免税
#         'level': 1, 'index_code': '801210', 
#         'etf_code': '515980', 'etf_name': '旅游ETF'
#     },

#     # ==============================
#     # Major Finance & Real Estate (Strongly correlated with government/policy)
#     # ==============================
#     '银行': {
#         'level': 1, 'index_code': '801192', 
#         'etf_code': '512800', 'etf_name': '银行ETF'
#     },
#     '非银金融': { # Securities + Insurance
#         'level': 1, 'index_code': '801193', 
#         'etf_code': '512070', 'etf_name': '证券ETF' # Securities have the highest volatility
#     },
#     '房地产': {
#         'level': 1, 'index_code': '801180', 
#         'etf_code': '512200', 'etf_name': '地产ETF'
#     },
#     '建筑装饰': { # Infrastructure
#         'level': 1, 'index_code': '801720', 
#         'etf_code': '516970', 'etf_name': '基建50ETF'
#     },

#     # ==============================
#     # Infrastructure
#     # ==============================
#     '公用事业': { # Power, environmental protection
#         'level': 1, 'index_code': '801160', 
#         'etf_code': '159611', 'etf_name': '电力ETF'
#     },
#     '交通运输': {
#         'level': 1, 'index_code': '801170', 
#         'etf_code': '159683', 'etf_name': '交运ETF'
#     },
#     '环保': {
#         'level': 1, 'index_code': '801970', 
#         'etf_code': '512580', 'etf_name': '环保ETF'
#     },

#     # ==============================
#     # Comprehensive
#     # ==============================
#     '综合': {
#         'level': 1, 'index_code': '801230', 
#         'etf_code': None, 'etf_name': None
#     },

#     # ========================================================
#     # High-value Level 2 Segments (Main source of alpha)
#     # If your advertising data can be precise to secondary industries, be sure to use the following
#     # ========================================================
    
#     # --- Electronics subdivision ---
#     '半导体': {
#         'level': 2, 'index_code': '801081', 
#         'etf_code': '512480', 'etf_name': '半导体ETF'
#     },
    
#     # --- Power Equipment subdivision ---
#     '光伏设备': {
#         'level': 2, 'index_code': '801735', 
#         'etf_code': '515790', 'etf_name': '光伏ETF'
#     },
#     '电池': {
#         'level': 2, 'index_code': '801736', 
#         'etf_code': '159755', 'etf_name': '电池ETF'
#     },

#     # --- Food & Beverage subdivision ---
#     '白酒II': {
#         'level': 2, 'index_code': '801123', 
#         'etf_code': '512690', 'etf_name': '酒ETF'
#     },
    
#     # --- Pharmaceuticals & Biotechnology subdivision ---
#     '医疗器械': {
#         'level': 2, 'index_code': '801153', 
#         'etf_code': '159883', 'etf_name': '医疗器械ETF'
#     },
#     '中药II': {
#         'level': 2, 'index_code': '801155', 
#         'etf_code': '560080', 'etf_name': '中药ETF'
#     },
    
#     # --- Media subdivision ---
#     '游戏II': {
#         'level': 2, 'index_code': '801766', 
#         'etf_code': '516010', 'etf_name': '游戏ETF'
#     },
    
#     # --- Agriculture, Forestry, Animal Husbandry & Fishery subdivision ---
#     '养殖业': { # pig cycle
#         'level': 2, 'index_code': '801015', 
#         'etf_code': '159865', 'etf_name': '养殖ETF'
#     }
# }

# Helper function: Get index code based on industry name from advertising data
# def get_target_by_sector(sector_name):
#     """
#     - Input: 'Industry' field from advertising data (e.g., '食品饮料', '半导体')
#     - Output: {'index': '801xxx', 'etf': '51xxxx'}
#     """
#     target = SW_SECTOR_MAPPING.get(sector_name)
#     if not target:
#         # Fuzzy matching logic (to handle minor name differences, e.g., '白酒' vs '白酒II')
#         for k, v in SW_SECTOR_MAPPING.items():
#             if k in sector_name or sector_name in k:
#                 return v
#         return None
#     return target

# SECTOR_ETF = {k: v['etf_code'] for k, v in SW_SECTOR_MAPPING.items()}

"\nDictionary structure description:\n\nKey: Industry name (industry field from your advertising data)\nValue: {\n    'level': Industry level (1=primary, 2=secondary),\n    'index_code': Shenwan index code (used for historical data backtesting with akshare),\n    'etf_code': Leading ETF code (used for actual trading),\n    'etf_name': ETF name (for manual verification)\n}\nNote: For industries without suitable ETFs, 'etf_code' is set to None.\n"

In [ ]:
def Download_SW_Index_Data(etf_folder_path, sector_code_dict, sector_path_mapping_path, Log_File_Path=""):
    """
    - Batch fetch Shenwan primary industry sector index data using AkShare
    - Due to official limitation, only full history data can be downloaded, not applicable for custom date
    - sector_code_dict is like: {'食品饮料': {'index_code': '801120'}, ...}
    """
    code_sector = {code: sector for sector, code in sector_code_dict.items()}
    # Check exist file of data (file format: "SW-index_code-begin_year-end_year.csv")
    exist_sector_code_list = []
    exist_code_list = []
    for filename in os.listdir(etf_folder_path):
        if filename.startswith("SW") and filename.endswith(".csv"):
            index_code = filename.split("-")[1]
            exist_sector_code_list.append((index_code, code_sector.get(index_code, "None")))
            exist_code_list.append(index_code)
    if exist_sector_code_list: THREAD_SAFE_PRINT("Download SW Index", f"Existing {len(exist_sector_code_list)} index data files found for codes: {exist_sector_code_list}", Log_File_Path)
    Check_File(sector_path_mapping_path)
    for index_code, sector_name in code_sector.items():
        try:
            # Historical data interface for Shenwan primary industry sectors
            # Note: AkShare API may be updated; this is the current standard interface
            # The symbol corresponds to the Shenwan index code, e.g., "801780"
            if index_code not in exist_code_list:
                THREAD_SAFE_PRINT("Download SW Index", f"Starting to fetch Shenwan {sector_name} with {index_code} index data...", Log_File_Path)
                df = ak.index_hist_sw(symbol=index_code, period="day")
                """
                The data is like:
                
                    代码	日期	收盘	开盘	最高	最低	成交量	成交额
                0	801780	2014-02-21	2049.03	2069.85	2076.96	2037.83	9.456221	62.312222
                1	801780	2014-02-24	1994.13	2031.78	2031.78	1983.27	12.891230	82.756811
                2	801780	2014-02-25	1969.95	1994.30	2007.30	1964.31	10.929277	67.945007
                3	801780	2014-02-26	1976.77	1960.40	1986.42	1960.39	8.541088	51.196589
                4	801780	2014-02-27	2005.24	1981.19	2020.41	1963.10	13.352829	83.444082
                """
                # Clean data
                df['日期'] = pd.to_datetime(df['日期'])
                df = df.rename(columns={'代码': 'index_code', '日期': 'date', '开盘': 'open', "收盘": "close", "最高": "high", "最低": "low", "成交量": "volume", "成交额": "turnover"})
                df = df.set_index('date') # Set Date as index
                df = df.sort_index() # Sort by Date
                start_date = df.index.min().strftime("%Y%m%d")
                end_date = df.index.max().strftime("%Y%m%d")
                # Save to CSV
                sw_sector_path_mapping = JsonFile_to_Dict(sector_path_mapping_path)
                csv_filename = f"SW-{index_code}-{start_date}-{end_date}.csv"
                sw_sector_path_mapping[sector_name] = csv_filename
                Dict_to_JsonFile(sw_sector_path_mapping, sector_path_mapping_path)
                df.to_csv(etf_folder_path + csv_filename)
                THREAD_SAFE_PRINT("Download SW Index", f"✅Successfully fetched: {sector_name} with {index_code} ({start_date} to {end_date})", Log_File_Path)
                THREAD_SAFE_PRINT("Download SW Index", f"Saved to: {csv_filename}", Log_File_Path)
                # Courtesy delay to avoid IP blocking
                time.sleep(2)
        except Exception as e:
            THREAD_SAFE_PRINT("Download SW Index", f"❌Failed to fetch {sector_name}: {e}", Log_File_Path)

# Mapping of Shenwan primary industry sectors to file path
SW_SECTOR_PATH_MAPPING_PATH = ETF_DATA_PATH + "SW-SECTOR-PATH-MAPPING.json"

# Fetch data
Download_SW_Index_Data(etf_folder_path=ETF_DATA_PATH, sector_code_dict=SW_SECTOR_INDEX, sector_path_mapping_path=SW_SECTOR_PATH_MAPPING_PATH)